# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdubakr77/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**Rule (plain words):** flag a page for review if it is both stale (not updated in a while) and still getting meaningful visibility, since a stale-but-visible page is losing ground on traffic it could still capture with a refresh.

**Reason codes:**
- `STALE_AND_VISIBLE`: page is old (180+ days since update) and still has meaningful impressions (500+)
- `STALE_LOW_VISIBILITY`: page is old but barely seen, lower priority
- `FRESH`: recently updated, not a candidate regardless of visibility

**Signal verdicts:**

- **Visibility (impressions_90d): CONFIRMED.** Decline rate rises consistently from 39% to 62% as impressions increase from the lowest to the mid-high bucket, supporting the idea that visible pages are not immune to decline, they are actually more likely to be declining, which is exactly the assumption behind flagging "stale and visible" pages as worth reviewing.

- **Staleness (days_since_last_update): MIXED.** The pattern is not consistent (90-180 days shows a higher decline rate than 180-365 days), and the two oldest buckets have very few rows (169 and 5), too small to draw a reliable conclusion. Staleness alone is not a clean signal in this data, it likely needs to be combined with visibility rather than used on its own.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Quick check on the two signals the rule leans on
staleness_check = df.groupby(pd.cut(df['days_since_last_update'], bins=[0, 90, 180, 365, 10000]), observed=True).agg(
    n=('trend_direction', 'size'),
    pct_declining=('trend_direction', lambda x: (x == 'down').mean())
)
print("Staleness vs decline:")
print(staleness_check)

visibility_check = df.groupby(pd.cut(df['impressions_90d'], bins=[0, 100, 500, 2000, 1000000]), observed=True).agg(
    n=('trend_direction', 'size'),
    pct_declining=('trend_direction', lambda x: (x == 'down').mean())
)
print("\nVisibility vs decline:")
print(visibility_check)

Staleness vs decline:
                            n  pct_declining
days_since_last_update                      
(0, 90]                 20655       0.512031
(90, 180]                9171       0.611057
(180, 365]                169       0.467456
(365, 10000]                5       0.600000

Visibility vs decline:
                     n  pct_declining
impressions_90d                      
(0, 100]          8006       0.389208
(100, 500]        5279       0.604281
(500, 2000]       6502       0.617964
(2000, 1000000]  10213       0.581416


## 2. Build the ranked queue (writes the CSV)

Encoding the rule: since visibility is the confirmed signal and staleness alone is unreliable, the score weights visibility more heavily, using staleness only as a secondary tiebreaker within visible pages.

Reviewing the top 20 ranked rows manually, one line each: the action, why it's flagged, and what would make this pick wrong.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np

# Score: primarily visibility-driven, staleness as secondary weight
df['visibility_score'] = np.log1p(df['impressions_90d'])
df['staleness_score'] = df['days_since_last_update'] / 365

df['action_score'] = (df['visibility_score'] * 0.7) + (df['staleness_score'] * 0.3)

def reason_code(row):
    if row['impressions_90d'] >= 500 and row['days_since_last_update'] >= 180:
        return "STALE_AND_VISIBLE"
    elif row['impressions_90d'] >= 500:
        return "VISIBLE_NOT_STALE"
    elif row['days_since_last_update'] >= 180:
        return "STALE_LOW_VISIBILITY"
    else:
        return "LOW_PRIORITY"

df['reason_code'] = df.apply(reason_code, axis=1)
df['action_label'] = df['reason_code'].apply(lambda x: "REVIEW" if x in ["STALE_AND_VISIBLE", "VISIBLE_NOT_STALE"] else "MONITOR")

ranked = df.sort_values('action_score', ascending=False)

output_cols = ['content_id', 'action_score', 'reason_code', 'action_label', 
               'impressions_90d', 'days_since_last_update', 'trend_direction']
ranked_output = ranked[output_cols] if 'content_id' in df.columns else ranked[[c for c in output_cols if c != 'content_id']]

import os
os.makedirs("../outputs", exist_ok=True)
ranked_output.to_csv("../outputs/baseline_action_score.csv", index=False)

print("Saved", len(ranked_output), "rows")
ranked_output.head(10)

Saved 30000 rows


,content_id,action_score,reason_code,action_label,impressions_90d,days_since_last_update,trend_direction
6653,content_5fe46e04994d,9.295507,VISIBLE_NOT_STALE,REVIEW,517715,104,down
17812,content_aaef01a50def,9.227290,VISIBLE_NOT_STALE,REVIEW,517109,22,stable
19636,content_2cb567c3c89b,9.221918,VISIBLE_NOT_STALE,REVIEW,497727,48,up
26844,content_8c19996aa890,9.214929,VISIBLE_NOT_STALE,REVIEW,509252,20,down
29400,content_2dba2b1f9536,9.187094,VISIBLE_NOT_STALE,REVIEW,443434,104,stable
21819,content_4c36c775b818,9.148433,VISIBLE_NOT_STALE,REVIEW,463103,20,down
29879,content_1a9e894be2e2,9.075295,VISIBLE_NOT_STALE,REVIEW,416180,22,down
13537,content_2c2606c5d176,9.016242,VISIBLE_NOT_STALE,REVIEW,347399,104,down
18870,content_db5989a78dd3,8.942575,VISIBLE_NOT_STALE,REVIEW,345111,20,up
26531,content_cb112fce36be,8.936308,VISIBLE_NOT_STALE,REVIEW,309910,104,down


In [6]:
top20 = ranked_output.head(20).reset_index(drop=True)
top20

,content_id,action_score,reason_code,action_label,impressions_90d,days_since_last_update,trend_direction
0,content_5fe46e04994d,9.295507,VISIBLE_NOT_STALE,REVIEW,517715,104,down
1,content_aaef01a50def,9.227290,VISIBLE_NOT_STALE,REVIEW,517109,22,stable
2,content_2cb567c3c89b,9.221918,VISIBLE_NOT_STALE,REVIEW,497727,48,up
3,content_8c19996aa890,9.214929,VISIBLE_NOT_STALE,REVIEW,509252,20,down
4,content_2dba2b1f9536,9.187094,VISIBLE_NOT_STALE,REVIEW,443434,104,stable
5,content_4c36c775b818,9.148433,VISIBLE_NOT_STALE,REVIEW,463103,20,down
6,content_1a9e894be2e2,9.075295,VISIBLE_NOT_STALE,REVIEW,416180,22,down
7,content_2c2606c5d176,9.016242,VISIBLE_NOT_STALE,REVIEW,347399,104,down
8,content_db5989a78dd3,8.942575,VISIBLE_NOT_STALE,REVIEW,345111,20,up
9,content_cb112fce36be,8.936308,VISIBLE_NOT_STALE,REVIEW,309910,104,down


## 3. Top-20 review

**Top-20 review:**

1. content_5fe46e04994d, score 9.30: REVIEW, VISIBLE_NOT_STALE. Trending down with 517K impressions, a strong genuine pick. Would be wrong if the impressions spike is seasonal noise rather than a sustained pattern.
2. content_aaef01a50def, score 9.23: REVIEW, VISIBLE_NOT_STALE. Trend is stable, not declining. This flags a healthy page just because it's visible, not because anything is wrong with it, likely a false positive for a "refresh" queue.
3. content_2cb567c3c89b, score 9.22: REVIEW, VISIBLE_NOT_STALE. Trend is actually trending up. This page needs no action at all, the score is rewarding visibility regardless of direction, this is a clear miss.
4. content_8c19996aa890, score 9.21: REVIEW, VISIBLE_NOT_STALE. Trending down with high impressions, a solid pick, matches the intent of the rule.
5. content_2dba2b1f9536, score 9.19: REVIEW, VISIBLE_NOT_STALE. Stable trend, another likely false positive, visibility alone triggered the flag.
6. content_4c36c775b818, score 9.15: REVIEW, VISIBLE_NOT_STALE. Trending down, good pick.
7. content_1a9e894be2e2, score 9.08: REVIEW, VISIBLE_NOT_STALE. Trending down, good pick.
8. content_2c2606c5d176, score 9.02: REVIEW, VISIBLE_NOT_STALE. Trending down, good pick, and stale at 104 days, closer to the intended "stale and visible" profile.
9. content_db5989a78dd3, score 8.94: REVIEW, VISIBLE_NOT_STALE. Trending up. Wrong pick, the page is improving, not declining.
10. content_cb112fce36be, score 8.94: REVIEW, VISIBLE_NOT_STALE. Trending down, stale at 104 days, good pick.
11. content_9532f197bbc8, score 8.93: REVIEW, VISIBLE_NOT_STALE. Trending down, stale, good pick.
12. content_36ff89c8214e, score 8.90: REVIEW, VISIBLE_NOT_STALE. Trend stable, likely false positive.
13. content_b28d1efd668f, score 8.88: REVIEW, VISIBLE_NOT_STALE. Trend stable, likely false positive.
14. content_44e481c8f55b, score 8.87: REVIEW, VISIBLE_NOT_STALE. Trend stable, likely false positive, and page is not even stale (20 days).
15. content_8e7ba84a972b, score 8.82: REVIEW, VISIBLE_NOT_STALE. Trend stable, not stale, likely false positive.
16. content_89e84d699e9e, score 8.78: REVIEW, VISIBLE_NOT_STALE. Trend stable, not stale, likely false positive.
17. content_8451fc6f034d, score 8.78: REVIEW, VISIBLE_NOT_STALE. Trend up, not stale. Wrong pick, page is improving.
18. content_813e88069237, score 8.74: REVIEW, VISIBLE_NOT_STALE. Trending down, stale at 104 days, good pick.
19. content_aa4baf490b43, score 8.73: REVIEW, VISIBLE_NOT_STALE. Trend stable, not stale, likely false positive.
20. content_008fb02c46cb, score 8.68: REVIEW, VISIBLE_NOT_STALE. Trending down, not stale, good pick.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

**Weak picks:** rows 2, 9, and 17 are trending up but still got flagged for review, this is the clearest failure of the rule: pure visibility weighting (0.7) drowns out the actual signal we care about (decline), so a healthy or improving page can outrank a genuinely declining one. About 35% of the top 20 (7 of 20) are stable or trending up, not down, meaning over a third of this "priority" list may waste review time on pages that don't need it.

**Leakage check:** the score uses only `impressions_90d` and `days_since_last_update`, both of which are observable before any decision is made. `trend_direction` (the label) was never used as a model input, it only appears in the output table for manual review, confirming no label leakage. No client names or product-level flags were used in scoring.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.